# Chapter 10B — Stage B: Feedback-Enhanced Netlist Generator

**Multi-Agent Analog EDA — Pedagogical Project (PhD track)**

---

This notebook implements **Stage B** of a two-stage pedagogical arc: given a **Component Inventory** artifact from **Stage A** (structured JSON describing allowed devices, bias budget, and performance targets), we **synthesize SKY130-grounded SPICE netlists** using **template compilation** and **AnalogCoder-style** program generation, then close a **feedback loop**—generate → (mock) simulate → score → retune parameters → regenerate—until metrics converge within specification.

### Learning objectives

1. **Formalize** netlist synthesis as *conditional code generation* $c \sim \pi(c \mid s, \mathcal{H})$ with compile-time checks (PDK bounds, connectivity) and runtime checks (simulator FoMs).
2. **Implement** **template-based** generators for **two-stage Miller OTA** and **folded-cascode OTA**, including **bias ladders** and a **AC/open-loop test bench**.
3. **Encode** **domain-specific prompts** (*speclets*) per sub-block and map them to Python **constructors** (PySpice-flavored composition).
4. **Build** a **closed-loop refiner** with a **deterministic mock oracle** $\tilde{\mathcal{O}}$ to study **convergence** without external EDA licenses.
5. **Enforce** **SKY130** conventions: model references, **W/L legality**, **five corners** (TT, FF, SS, SF, FS), and **temperature** windows.
6. **Visualize** metric and parameter trajectories plus **netlist diffs** across iterations.

### Notation

- Structured inventory $s \in \mathcal{S}$ (Stage A JSON).
- Executable netlist string $c \in \mathcal{C}$ (SPICE text + `.lib` hooks).
- Mock metrics $\phi = \tilde{\mathcal{O}}(c, \theta)$ with parameters $\theta$ (widths, lengths, bias currents).
- Iteration history $\mathcal{H}_t = \{(c_i, \phi_i, \theta_i)\}_{i \le t}$.

> **Disclaimer:** The **simulator is mocked** for portability. Swap `mock_simulate` for Ngspice/Xyce subprocess calls while retaining the same interface.

---


In [ ]:
import sys; sys.path.insert(0, '..')
from style_utils import (setup_3b1b_style, glow_line, glow_fill, styled_box,
                         styled_arrow, finish_plot, plotly_3b1b_layout,
                         BACKGROUND, SURFACE, TEXT, TEXT_DIM, GRID,
                         BLUE, TEAL, GREEN, YELLOW, GOLD, RED,
                         ROSE, PURPLE, CYAN, ORANGE, PALETTE)
setup_3b1b_style()

# Imports, RNG, and dark-theme defaults (matplotlib #0d1117, plotly plotly_dark)
from __future__ import annotations

import difflib
import json
import math
import re
import textwrap
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt

import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

RNG = np.random.default_rng(2026)

DARK_BG = "#0d1117"
ACCENT = "#58a6ff"
GREEN = "#3fb950"
AMBER = "#d29922"
RED = "#f85149"
PURPLE = "#bc8cff"
CYAN = "#39d353"

MPL_RC = {
    "figure.facecolor": DARK_BG,
    "axes.facecolor": DARK_BG,
    "axes.edgecolor": "#30363d",
    "axes.labelcolor": "#c9d1d9",
    "text.color": "#c9d1d9",
    "xtick.color": "#8b949e",
    "ytick.color": "#8b949e",
    "grid.color": "#21262d",
    "grid.alpha": 0.65,
    "legend.facecolor": "#161b22",
    "legend.edgecolor": "#30363d",
    "font.size": 11,
}
mpl.rcParams.update(MPL_RC)
pio.templates.default = "plotly_dark"

print("Environment: numpy; matplotlib facecolor", DARK_BG, "; plotly template", pio.templates.default)


## 10B.1 Problem statement — from inventory to synthesizable netlists

**Input:** A **Component Inventory JSON** $\texttt{inventory}$ emitted by Stage A. It enumerates:

- Allowed **SKY130** primitive flavors (`nfet_01v8`, `pfet_01v8`, …) and **global bias** budget.
- **Topology choice** or a ranked set (`miller_ota` vs `folded_cascode_ota`).
- **Targets** $(A_{\mathrm{dc}}, \mathrm{GBW}, \mathrm{PM}, I_{\mathrm{tot}})$ and **verification corners**.

**Output:** A **single `.spice` artifact** (string) that:

1. **Includes** PDK libraries and uses **legal device instances** (`X` + correct `sky130_fd_pr` cells).
2. **Declares** the OTA subcircuit, **bias network**, and **test bench** (stimulus + probes).
3. Is **revisable** through parameters $\theta$ (W/L, `ibias`, Miller cap) so a **closed-loop agent** can iterate.

**Closed-loop objective:** minimize structured loss

$$
\mathcal{J}(\theta) = \sum_k w_k \left\lvert \frac{\phi_k(\theta) - \phi_k^\star}{\phi_k^\star} \right\rvert^2 + \lambda \, \mathrm{viol}(\theta),
$$

where $\mathrm{viol}$ aggregates PDK bound violations (zero if feasible).

We instantiate $\texttt{inventory}$ below as a **self-contained** example you can edit.


In [ ]:
# Stage A — Component Inventory (representative JSON)
INVENTORY: Dict[str, Any] = {
    "schema": "multi_agent_analog_eda/component_inventory/v1",
    "pdk": "sky130A",
    "library_hooks": {
        # User must set SKY130_ROOT; paths vary by install — kept symbolic for pedagogy
        "lib_file": "${SKY130_ROOT}/libs.tech/ngspice/sky130.lib.spice",
        "corner_default": "tt",
    },
    "allowed_devices": [
        {"cell": "sky130_fd_pr__nfet_01v8", "role": "nmos", "notes": "LV NFET"},
        {"cell": "sky130_fd_pr__pfet_01v8", "role": "pmos", "notes": "LV PFET"},
    ],
    "topology_preferences": ["miller_ota", "folded_cascode_ota"],
    "electrical_targets": {
        "Adc_dB": 72.0,
        "GBW_MHz": 12.0,
        "phase_margin_deg": 62.0,
        "Itot_uA": 180.0,
    },
    "verification": {
        "corners": ["tt", "ff", "ss", "sf", "fs"],
        "temperature_C": {"min": -40, "typ": 27, "max": 125},
    },
}

print(json.dumps(INVENTORY, indent=2))


## 10B.2 SKY130 PDK compliance — models, geometry, corners, temperature

**Device models:** Ngspice/Xyce decks typically include a **cornered library** such as `sky130.lib.spice` and instantiate transistors as subcircuit `X` elements, e.g. `sky130_fd_pr__nfet_01v8` / `sky130_fd_pr__pfet_01v8`.

**Geometry bounds (pedagogical guardrails):** SKY130 LV FETs are often drafted with **minimum channel length** $L \approx 0.15\,\mu$m and **minimum width** on the order of **hundreds of nm** depending on flavor and DRC deck. We embed **conservative** bounds (tunable) and reject illegal $(W,L)$ at **compile time**.

**Process corners:** We support the canonical **five** MOS corners:

| Corner | NFET / PFET skew (conceptual) |
|--------|--------------------------------|
| **TT** | Typical / Typical |
| **FF** | Fast / Fast |
| **SS** | Slow / Slow |
| **SF** | Slow NFET / Fast PFET |
| **FS** | Fast NFET / Slow PFET |

**Temperature:** Automotive / industrial specs often require **$-40^\circ$C to $125^\circ$C**. We emit **`.temp`** / `.option temp` lines per analysis block in generated decks.

The next cell encodes **validators** and **corner/temp header** synthesis.


In [ ]:
# SKY130-style bounds and compliance helpers (educational defaults)
@dataclass(frozen=True)
class Sky130LVBounds:
    W_min_um: float = 0.42
    W_max_um: float = 100.0
    L_min_um: float = 0.15
    L_max_um: float = 10.0


SKY130_LV = Sky130LVBounds()
CORNER_ALIASES = {"tt", "ff", "ss", "sf", "fs"}


def check_wl_um(W_um: float, L_um: float, bounds: Sky130LVBounds = SKY130_LV) -> Tuple[bool, List[str]]:
    errs: List[str] = []
    if not (bounds.W_min_um <= W_um <= bounds.W_max_um):
        errs.append(f"W={W_um} µm outside [{bounds.W_min_um}, {bounds.W_max_um}]")
    if not (bounds.L_min_um <= L_um <= bounds.L_max_um):
        errs.append(f"L={L_um} µm outside [{bounds.L_min_um}, {bounds.L_max_um}]")
    return (len(errs) == 0), errs


def assert_corners(corners: List[str]) -> None:
    bad = [c for c in corners if c.lower() not in CORNER_ALIASES]
    if bad:
        raise ValueError(f"Unsupported corners: {bad}; expected subset of {sorted(CORNER_ALIASES)}")


def temp_window_ok(tmin: float, tmax: float) -> Tuple[bool, str]:
    # Industry-style automotive window check
    ok = tmin <= -40 + 1e-9 and tmax >= 125 - 1e-9
    return ok, ("PASS: spans [-40,125]°C" if ok else "NOTE: inventory window narrower than [-40,125]°C")


def spice_corner_banner(corners: List[str], lib_file: str, temp_C: float) -> str:
    assert_corners(corners)
    lines = [
        "* SKY130 corner sweep scaffold — uncomment .STEP or repeat runs per corner",
        f".lib {lib_file} tt",
        f".temp {temp_C}",
        ".option SCALE=1.0",
        ".option ABSTOL=1e-12 RELTOL=1e-3 VNTOL=1e-6",
        "",
        "* Example multi-corner orchestration (Ngspice):",
        "* .STEP dec temp list -40 27 125",
        "* For true FF/SS/SF/FS, re-point .lib to the appropriate section per PDK docs.",
    ]
    return "\n".join(lines)


ver = INVENTORY["verification"]
tw_ok, tw_msg = temp_window_ok(ver["temperature_C"]["min"], ver["temperature_C"]["max"])
print("Corner set:", ver["corners"])
print("Temperature window check:", tw_msg)
print(spice_corner_banner(ver["corners"], "${SKY130_ROOT}/libs.tech/ngspice/sky130.lib.spice", ver["temperature_C"]["typ"]))


## 10B.3 AnalogCoder principles — programs as analog artifacts

**Core idea (AnalogCoder):** analog design is **code synthesis** under **executable semantics**. Instead of hand-drawing every instance, we:

1. **Factor** knowledge into **reusable Python constructors** (a shallow embedded DSL).
2. Attach **domain-specific prompts** (*speclets*) per block—textual constraints an LLM (or human) must respect when editing generators.
3. **Compile** Python structures → **SPICE strings** (*code-to-netlist compilation*), analogous to PySpice’s programmatic deck construction but **template-first** for reliability.
4. **Close the loop** with $\tilde{\mathcal{O}}$ or a real simulator: **execution feedback** drives the next program revision.

We implement **speclets** as structured strings and a **`NetlistCompiler`** that:

- Validates W/L against `Sky130LVBounds`.
- Emits **`.subckt`**, instances, **bias**, and **testbench** blocks.
- Returns **`compile_report`** metadata for agents (violations, parameters).

> **PySpice-style** here means *compositional*, *type-ish* helpers (`mosfet_line`, `vdc`, `iac`)—not an import of proprietary decks.


In [ ]:
# Domain-specific prompts (speclets) per circuit block — used by human/LLM editors of this notebook
SPECLETS: Dict[str, str] = {
    "bias_nmos": textwrap.dedent(
        """
        Bias NMOS tail / cascode bias:
        - Mirror topology must keep all branches in saturation under TT.
        - Keep Vds headroom >= 0.1V typical per stacked device unless overriden by sim.
        - Use sky130_fd_pr__nfet_01v8; bulk=tie to lowest NMOS rail unless DNW.
        """
    ).strip(),
    "bias_pmos": textwrap.dedent(
        """
        PMOS current sources / loads:
        - Prefer self-cascode or long-L if output impedance must rise without huge W.
        - Keep |Vsg| within 1v8 nominal; confirm gds drop vs target GBW.
        """
    ).strip(),
    "miller_stage": textwrap.dedent(
        """
        Two-stage Miller OTA:
        - Pole-splitting capacitor Cc from second-stage gate to output (non-inverting second stage).
        - Nulling resistor Rz optional (series with Cc) — omit here for brevity.
        - Enforce phase margin via Cc and relative gm ratios (gm2 > gm1 typically helps GBW).
        """
    ).strip(),
    "folded_cascode": textwrap.dedent(
        """
        Folded-cascode OTA:
        - PFET input pair folded into NMOS cascode stack (this template polarity).
        - Watch headroom on folding node; keep cascode gates biased mid-rail.
        - High-swing bias generation not shown — use ideal references for pedagogy.
        """
    ).strip(),
    "tb_ac": textwrap.dedent(
        """
        Open-loop AC test bench:
        - Break loop with huge inductor / huge capacitor 'Analog EDA textbook' method.
        - Stimulate with small AC source at input; probe output.
        """
    ).strip(),
}

for k, v in SPECLETS.items():
    print(f"=== {k} ===\n{v}\n")


In [ ]:
# PySpice-flavor compositional helpers + compiler shell

def comment_block(title: str) -> str:
    return f"\n* --- {title} ---\n"


def xmos(name: str, d: str, g: str, s: str, b: str, cell: str, w_um: float, l_um: float, **extras: float) -> str:
    ok, errs = check_wl_um(w_um, l_um)
    if not ok:
        raise ValueError(f"{name}: " + "; ".join(errs))
    extra = ""
    if extras:
        parts = [f"{k}={v}" for k, v in extras.items()]
        extra = " " + " ".join(parts)
    return f"X{name} {d} {g} {s} {b} {cell} w={w_um}u l={l_um}u{extra}\n"


def idev(name: str, nplus: str, nminus: str, val: float) -> str:
    return f"I{name} {nplus} {nminus} DC {val}\n"


def vdc(name: str, nplus: str, nminus: str, v: float) -> str:
    return f"V{name} {nplus} {nminus} DC {v}\n"


def vac(name: str, nplus: str, nminus: str, ac_mag: float) -> str:
    return f"V{name} {nplus} {nminus} DC 0 AC {ac_mag}\n"


def iac(name: str, nplus: str, nminus: str, ac_mag: float) -> str:
    return f"I{name} {nplus} {nminus} DC 0 AC {ac_mag}\n"


def ldev(name: str, n1: str, n2: str, l_h: float) -> str:
    return f"L{name} {n1} {n2} {l_h}\n"


def cdev(name: str, n1: str, n2: str, c_f: float) -> str:
    return f"C{name} {n1} {n2} {c_f}\n"


@dataclass
class CompileReport:
    topology: str
    params: Dict[str, float]
    violations: List[str] = field(default_factory=list)
    speclets_used: List[str] = field(default_factory=list)


class NetlistCompiler:
    """Template compiler: parameters -> SPICE string + report."""

    def __init__(self, inventory: Dict[str, Any]):
        self.inv = inventory
        self.lib_file = inventory["library_hooks"]["lib_file"]
        self.corners = inventory["verification"]["corners"]
        self.temp_typ = float(inventory["verification"]["temperature_C"]["typ"])

    def preamble(self) -> str:
        return (
            "* Autogenerated pedagogical SKY130 deck (Stage B)\n"
            f"* Topology compiler — corners requested: {', '.join(self.corners)}\n"
            + spice_corner_banner(self.corners, self.lib_file, self.temp_typ)
            + "\n"
        )

    def miller_two_stage(self, p: Dict[str, float]) -> Tuple[str, CompileReport]:
        """
        Two-stage Miller OTA (simplified):
          - NMOS differential pair (M1,M2), PMOS mirror load (M3,M4)
          - Second stage: NMOS common-source (M6) with PMOS load (M5)
          - Miller cap Cc on gate of M6
        Nodes are symbolic; bias via ideal current source IBIAS.
        """
        W1, L1 = p["W_in_um"], p["L_in_um"]
        W5, L5 = p["W_load_um"], p["L_load_um"]
        W6, L6 = p["W_cs_um"], p["L_cs_um"]
        Cc = p["Cc_pF"]
        ibias = p["ibias_A"]

        s = self.preamble()
        s += comment_block("Speclet: miller_stage") + f"* {SPECLETS['miller_stage'].replace(chr(10), chr(10)+'* ')}\n"
        s += comment_block("Subcircuit miller_ota")
        s += ".subckt miller_ota vip vin vout vdd vss\n"
        s += idev("BIAS", "nbias", "vss", ibias)
        # diff pair
        s += xmos("1", "n1", "vip", "ntail", "vss", "sky130_fd_pr__nfet_01v8", W1, L1)
        s += xmos("2", "n2", "vin", "ntail", "vss", "sky130_fd_pr__nfet_01v8", W1, L1)
        s += xmos("TAIL", "ntail", "nbias", "vss", "vss", "sky130_fd_pr__nfet_01v8", 2 * W1, L1)
        # pmos load + mirror
        s += xmos("3", "n1", "n1", "vdd", "vdd", "sky130_fd_pr__pfet_01v8", 2 * W1, L1)
        s += xmos("4", "n2", "n1", "vdd", "vdd", "sky130_fd_pr__pfet_01v8", 2 * W1, L1)
        # second stage
        s += xmos("5", "vout", "n2", "vdd", "vdd", "sky130_fd_pr__pfet_01v8", W5, L5)
        s += xmos("6", "vout", "n3", "vss", "vss", "sky130_fd_pr__nfet_01v8", W6, L6)
        s += cdev("C", "n3", "vout", Cc * 1e-12)
        s += ".ends miller_ota\n"

        s += comment_block("Bias / references") + f"* {SPECLETS['bias_nmos']}\n"
        s += comment_block("Testbench — open-loop AC")
        s += f"* {SPECLETS['tb_ac']}\n"
        s += ".subckt tb_miller vip vin vdd vss\n"
        s += vdc("DD", "vdd", "vss", 1.8)
        s += vac("IN", "vip", "vss", 1.0)
        s += vdc("CM", "vin", "vss", 0.9)
        s += ldev("L1", "vip", "vin", 1e9)
        s += cdev("C1", "vin", "vss", 1e9)
        s += "Xm dut vip vin vo vdd vss miller_ota\n"
        s += ".ends tb_miller\n"
        s += "Xtb vip vin vdd vss tb_miller\n"
        s += ".ac dec 100 1 1e9\n"
        s += ".probe ac vm(vo) vp(vo)\n"
        s += ".end\n"

        rep = CompileReport(
            topology="miller_ota",
            params=p,
            speclets_used=["miller_stage", "bias_nmos", "tb_ac"],
        )
        return s, rep

    def folded_cascode(self, p: Dict[str, float]) -> Tuple[str, CompileReport]:
        """
        Folded cascode OTA skeleton:
          PFET diff pair -> folding node -> NMOS cascode -> output
          PMOS cascode stack as load
        """
        Wp_in, Lp_in = p["W_pin_um"], p["L_pin_um"]
        Wn_fc, Ln_fc = p["W_ncas_um"], p["L_ncas_um"]
        Wp_ld, Lp_ld = p["W_pload_um"], p["L_pload_um"]
        ibias = p["ibias_A"]

        s = self.preamble()
        s += comment_block("Speclet: folded_cascode") + f"* {SPECLETS['folded_cascode'].replace(chr(10), chr(10)+'* ')}\n"
        s += ".subckt folded_ota vip vin vout vdd vss\n"
        s += idev("FOLD", "nf", "vss", ibias)
        s += idev("PLOAD", "vdd", "nload", ibias)
        # input PFET pair
        s += xmos("INP", "nf", "vip", "nsrc", "vdd", "sky130_fd_pr__pfet_01v8", Wp_in, Lp_in)
        s += xmos("INN", "n1", "vin", "nsrc", "vdd", "sky130_fd_pr__pfet_01v8", Wp_in, Lp_in)
        s += xmos("SRC", "nsrc", "nbp", "vdd", "vdd", "sky130_fd_pr__pfet_01v8", 2 * Wp_in, Lp_in)
        # NMOS folding + cascode
        s += xmos("FN1", "n1", "nf", "vss", "vss", "sky130_fd_pr__nfet_01v8", 2 * Wp_in, Lp_in)
        s += xmos("NCAS", "vout", "ngn", "n1", "vss", "sky130_fd_pr__nfet_01v8", Wn_fc, Ln_fc)
        # PMOS load cascode
        s += xmos("PCAS", "vout", "pgp", "nload", "vdd", "sky130_fd_pr__pfet_01v8", Wp_ld, Lp_ld)
        s += xmos("PLD", "nload", "nload", "vdd", "vdd", "sky130_fd_pr__pfet_01v8", Wp_ld, Lp_ld)
        # ideal bias nodes as DC voltages for pedagogy
        s += vdc("BGN", "ngn", "vss", p["Vcasn"])
        s += vdc("BGP", "pgp", "vdd", p["Vcasp"])
        s += vdc("NBP", "nbp", "vss", p["Vbiasp"])
        s += ".ends folded_ota\n"

        s += comment_block("Testbench — folded OTA AC")
        s += ".subckt tb_folded vip vin vdd vss\n"
        s += vdc("DD", "vdd", "vss", 1.8)
        s += vac("IN", "vip", "vss", 1.0)
        s += vdc("CM", "vin", "vss", 0.9)
        s += ldev("L1", "vip", "vin", 1e9)
        s += cdev("C1", "vin", "vss", 1e9)
        s += "Xf dut vip vin vo vdd vss folded_ota\n"
        s += ".ends tb_folded\n"
        s += "Xtf vip vin vdd vss tb_folded\n"
        s += ".ac dec 100 1 1e9\n"
        s += ".probe ac vm(vo) vp(vo)\n"
        s += ".end\n"

        rep = CompileReport(
            topology="folded_cascode_ota",
            params=p,
            speclets_used=["folded_cascode", "bias_pmos", "bias_nmos", "tb_ac"],
        )
        return s, rep


compiler = NetlistCompiler(INVENTORY)
print("NetlistCompiler ready.")


In [ ]:
# Example: emit complete .spice strings for both topologies
miller_params = {
    "W_in_um": 2.0,
    "L_in_um": 0.15,
    "W_load_um": 4.0,
    "L_load_um": 0.3,
    "W_cs_um": 6.0,
    "L_cs_um": 0.15,
    "Cc_pF": 1.2,
    "ibias_A": 25e-6,
}

folded_params = {
    "W_pin_um": 3.0,
    "L_pin_um": 0.15,
    "W_ncas_um": 4.0,
    "L_ncas_um": 0.15,
    "W_pload_um": 4.0,
    "L_pload_um": 0.15,
    "ibias_A": 30e-6,
    "Vcasn": 0.65,
    "Vcasp": 1.1,
    "Vbiasp": 0.55,
}

sp_miller, rep_m = compiler.miller_two_stage(miller_params)
sp_fold, rep_f = compiler.folded_cascode(folded_params)

print("--- Miller deck (first 40 lines) ---")
print("\n".join(sp_miller.splitlines()[:40]))
print("...")
print("\n--- Folded deck (first 40 lines) ---")
print("\n".join(sp_fold.splitlines()[:40]))
print("...")


## 10B.4 Feedback loop — mock simulation, metrics, parameter updates

We realize the **agentic refinement cycle**:

```
  θ₀ → compile(c₀) → φ̃₀ = Õ(c₀,θ₀) → score J₀
         ↑                              |
         └──── Δθ(φ̃,φ*) ────────────────┘
```

**Mock oracle $\tilde{\mathcal{O}}$ (transparent):** we use smooth **surrogate** maps motivated by first-order MOS small-signal scaling:

- $g_m \propto \sqrt{W/L \cdot I_D}$ (square-law intuition),
- DC gain $A_{\mathrm{dc}} \propto g_m^2 \cdot r_o$ (very coarse),
- Dominant Miller pole $\propto 1/(g_{m2} R_2 C_c)$ → **GBW** $\approx g_{m1}/C_c$ textbook,
- **Phase margin** increases with **larger** $C_c$ and **lower** second-stage $g_m$ (again, coarse).

The goal is not SPICE accuracy but **a smooth surrogate landscape** for pedagogy: with normalized gradient steps, the weighted loss $\mathcal{J}$ typically **falls over iterations** while metrics **track toward** the Stage A targets.

**Update rule:** projected **gradient-like** step on a few **macro-knobs** (W factors, `Cc`, `ibias`) with **clipping** to the SKY130 window.


In [ ]:
@dataclass
class Metrics:
    Adc_dB: float
    GBW_MHz: float
    PM_deg: float
    Itot_uA: float


def mock_simulate_miller(p: Dict[str, float], targets: Dict[str, float]) -> Metrics:
    """Deterministic surrogate: sqrt-law drives + Miller Cc roll-off (pedagogy scale, not SPICE-exact)."""
    W1, L1 = p["W_in_um"], p["L_in_um"]
    W5, L5 = p["W_load_um"], p["L_load_um"]
    W6, L6 = p["W_cs_um"], p["L_cs_um"]
    Cc = max(p["Cc_pF"], 0.1)
    ib = max(p["ibias_A"], 1e-9)

    drive = math.sqrt(max((W1 / L1) * ib * 1e6, 1e-12))
    s2 = math.sqrt(max((W6 / L6) * ib * 1e6, 1e-12))
    load = math.sqrt(max(W5 / L5, 1e-12))

    GBW_MHz = 12.0 * (drive / 18.0) ** 0.55 * (1.2 / Cc) ** 0.65 / (1.0 + 0.04 * (load - 3.5) ** 2)

    Ad_lin = min(12000.0, 2000.0 * (drive / 15.0) ** 1.45 * (s2 / 28.0) ** 1.15 / (Cc ** 0.38))
    Adc_dB = 20 * math.log10(max(Ad_lin, 1e-3))

    PM_deg = (
        62.0
        + 8.0 * math.tanh(Cc - 1.15)
        - 6.0 * math.tanh((ib * 1e6 - 40.0) / 80.0)
        - 4.0 * math.tanh(s2 / 45.0 - 0.7)
    )
    PM_deg = float(max(38.0, min(PM_deg, 78.0)))

    Itot_uA = ib * 1e6 * 4.8 + 10.5 + 1.8 * load
    return Metrics(Adc_dB, GBW_MHz, PM_deg, Itot_uA)


def loss_metrics(m: Metrics, t: Dict[str, float], w: Optional[Dict[str, float]] = None) -> float:
    w = w or {"Adc_dB": 1.0, "GBW_MHz": 1.2, "phase_margin_deg": 0.9, "Itot_uA": 0.4}

    def rel(x, x0):
        return ((x - x0) / max(abs(x0), 1e-6)) ** 2

    return (
        w["Adc_dB"] * rel(m.Adc_dB, t["Adc_dB"])
        + w["GBW_MHz"] * rel(m.GBW_MHz, t["GBW_MHz"])
        + w["phase_margin_deg"] * rel(m.PM_deg, t["phase_margin_deg"])
        + w["Itot_uA"] * rel(m.Itot_uA, t["Itot_uA"])
    )


def refine_miller_params(p: Dict[str, float], m: Metrics, t: Dict[str, float], eta: float) -> Dict[str, float]:
    """Finite-difference sensitivities → parameter nudge (projected)."""
    base = loss_metrics(m, t)
    keys = ["W_in_um", "W_cs_um", "W_load_um", "Cc_pF", "ibias_A"]
    grads = {}
    for k in keys:
        q = dict(p)
        dq = 0.03 * q[k] if k != "Cc_pF" else 0.05
        q[k] = q[k] + dq
        if k == "ibias_A":
            q[k] = min(q[k], 120e-6)
        m2 = mock_simulate_miller(q, t)
        grads[k] = (loss_metrics(m2, t) - base) / dq

    gvec = np.array([grads[k] for k in keys], dtype=float)
    gn = float(np.linalg.norm(gvec) + 1e-12)
    newp = dict(p)
    for k in keys:
        newp[k] = newp[k] - (eta * grads[k]) / gn

    # Project to bounds / sanity
    newp["W_in_um"] = float(np.clip(newp["W_in_um"], SKY130_LV.W_min_um, 20.0))
    newp["W_cs_um"] = float(np.clip(newp["W_cs_um"], SKY130_LV.W_min_um, 40.0))
    newp["W_load_um"] = float(np.clip(newp["W_load_um"], SKY130_LV.W_min_um, 40.0))
    newp["Cc_pF"] = float(np.clip(newp["Cc_pF"], 0.2, 5.0))
    newp["ibias_A"] = float(np.clip(newp["ibias_A"], 5e-6, 120e-6))
    for lk in ["L_in_um", "L_load_um", "L_cs_um"]:
        newp[lk] = p[lk]
    return newp


def run_refinement_loop(
    p0: Dict[str, float],
    targets: Dict[str, float],
    iterations: int = 18,
    eta0: float = 0.35,
) -> Dict[str, Any]:
    hist = {"loss": [], "metrics": [], "params": [], "netlists": []}
    p = dict(p0)
    compiler_local = NetlistCompiler(INVENTORY)
    for it in range(iterations):
        sp, rep = compiler_local.miller_two_stage(p)
        m = mock_simulate_miller(p, targets)
        J = loss_metrics(m, targets)
        hist["loss"].append(J)
        hist["metrics"].append(m)
        hist["params"].append(dict(p))
        hist["netlists"].append(sp)
        eta = eta0 * (1.0 / (1.0 + 0.12 * it))
        p = refine_miller_params(p, m, targets, eta=eta)
    return hist


targets = INVENTORY["electrical_targets"]
seed = {
    "W_in_um": 1.2,
    "L_in_um": miller_params["L_in_um"],
    "W_load_um": 2.2,
    "L_load_um": miller_params["L_load_um"],
    "W_cs_um": 3.4,
    "L_cs_um": miller_params["L_cs_um"],
    "Cc_pF": 0.42,
    "ibias_A": 95e-6,
}
history = run_refinement_loop(seed, targets, iterations=24, eta0=2.8)

print("Initial loss:", history["loss"][0], "Final loss:", history["loss"][-1])
print("Initial metrics:", history["metrics"][0])
print("Final metrics:  ", history["metrics"][-1])


In [ ]:
# Matplotlib — convergence of loss + normalized metrics (dark #0d1117)
loss = np.array(history["loss"])
iters = np.arange(len(loss))

Adc = np.array([m.Adc_dB for m in history["metrics"]])
GBW = np.array([m.GBW_MHz for m in history["metrics"]])
PM = np.array([m.PM_deg for m in history["metrics"]])
IuA = np.array([m.Itot_uA for m in history["metrics"]])

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(iters, loss, color=ACCENT, lw=2, marker="o", ms=3, label=r"$\mathcal{J}(\theta_t)$")
axes[0].set_ylabel("Weighted relative MSE")
axes[0].set_title("Feedback loop — mock-simulation loss convergence")
axes[0].legend(loc="upper right", framealpha=0.9)
axes[0].grid(True)


def norm_track(x, x0):
    return x / max(abs(x0), 1e-6)


axes[1].plot(iters, norm_track(Adc, targets["Adc_dB"]), label=r"$A_{dc}$ (norm.)", color=GREEN)
axes[1].plot(iters, norm_track(GBW, targets["GBW_MHz"]), label="GBW (norm.)", color=PURPLE)
axes[1].plot(iters, norm_track(PM, targets["phase_margin_deg"]), label="PM (norm.)", color=AMBER)
axes[1].plot(iters, norm_track(IuA, targets["Itot_uA"]), label=r"$I_{tot}$ (norm.)", color=CYAN)
axes[1].axhline(1.0, color="#8b949e", ls="--", lw=1)
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Metric / target")
axes[1].set_title("Figures of merit approaching specification (mock)")
axes[1].legend(ncol=2, framealpha=0.9)
axes[1].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Matplotlib — parameter trajectories
keys = ["W_in_um", "W_cs_um", "W_load_um", "Cc_pF", "ibias_A"]
mat = {k: np.array([h[k] for h in history["params"]]) for k in keys}

fig, axes = plt.subplots(len(keys), 1, figsize=(10, 9), sharex=True)
colors = [ACCENT, GREEN, PURPLE, AMBER, CYAN]
for ax, k, c in zip(axes, keys, colors):
    ax.plot(iters, mat[k], color=c, lw=2)
    ylab = k.replace("_um", " (µm)").replace("Cc_pF", "Cc (pF)").replace("ibias_A", "ibias (A)")
    ax.set_ylabel(ylab, fontsize=10)
    ax.grid(True)
axes[-1].set_xlabel("Iteration")
fig.suptitle("Parameter evolution under compile–simulate–score–update", y=1.01, color="#c9d1d9")
plt.tight_layout()
plt.show()


In [ ]:
# Plotly — interactive convergence dashboard (template plotly_dark)
figp = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(r"Loss $\mathcal{J}$", "Key parameters"),
    vertical_spacing=0.12,
)

figp.add_trace(
    go.Scatter(x=iters, y=loss, mode="lines+markers", name="Loss", line=dict(color=ACCENT)),
    row=1,
    col=1,
)
figp.add_trace(
    go.Scatter(x=iters, y=mat["W_in_um"], name="W_in", line=dict(color=GREEN)), row=2, col=1
)
figp.add_trace(
    go.Scatter(x=iters, y=mat["Cc_pF"], name="Cc", line=dict(color=PURPLE)), row=2, col=1
)
figp.add_trace(
    go.Scatter(x=iters, y=mat["ibias_A"] * 1e6, name="ibias (µA)", line=dict(color=AMBER)), row=2, col=1
)

figp.update_layout(
    template="plotly_dark",
    paper_bgcolor=DARK_BG,
    plot_bgcolor="#161b22",
    height=640,
    title="Stage B — interactive refinement trace",
    legend_orientation="h",
    margin=dict(t=70, b=50),
)
figp.update_xaxes(gridcolor="#21262d", zeroline=False)
figp.update_yaxes(gridcolor="#21262d", zeroline=False)
figp.show()


In [ ]:
# Netlist diff between early and late iterations (unified diff)
i0, i1 = 0, len(history["netlists"]) - 1
a = history["netlists"][i0].splitlines(keepends=True)
b = history["netlists"][i1].splitlines(keepends=True)

diff_lines = list(difflib.unified_diff(a, b, fromfile=f"iter{i0}.spice", tofile=f"iter{i1}.spice", n=2))
print(f"Unified diff (iteration {i0} → {i1}) — {len(diff_lines)} lines")
print("".join(diff_lines[:120]))
if len(diff_lines) > 120:
    print(f"... ({len(diff_lines) - 120} more lines)")

# Highlight: parameter edits are concentrated in X-lines and passive values
pat = re.compile(r"(w=|l=|DC |C\w+ \S+ \S+ )")
changed = [ln for ln in diff_lines if ln.startswith(("+", "-")) and pat.search(ln)]
print("\n--- Diff lines touching W/L/bias/C (sample) ---")
print("".join(changed[:40]))


## 10B.5 Multi-corner & temperature harness (generation sketch)

Real sign-off requires **re-running** the generated deck under each **corner** and **temperature**. A robust pattern is:

1. **Emit** a *single* parameterized subcircuit netlist (above).
2. **Wrap** it with a **driver script** (Python/bash) that iterates `(corner, temp)` and swaps the `.lib` section per SKY130 documentation.
3. Aggregate $\phi$ into **worst-case** statistics for the agent (e.g., min PM across corners).

Below we **synthesize** a small **corner manifest** JSON from the inventory—suitable for a downstream **multi-agent orchestrator** to schedule parallel simulations.


In [ ]:
def corner_manifest(inv: Dict[str, Any]) -> Dict[str, Any]:
    temps = inv["verification"]["temperature_C"]
    corners = inv["verification"]["corners"]
    jobs = []
    for cor in corners:
        for T in (temps["min"], temps["typ"], temps["max"]):
            jobs.append(
                {
                    "corner": cor.lower(),
                    "temp_C": T,
                    "lib_stanza": f".lib {inv['library_hooks']['lib_file']} {cor.lower()}",
                    "notes": "Re-point stanza per installed PDK; some flows use include chains.",
                }
            )
    return {"jobs": jobs, "count": len(jobs)}


manifest = corner_manifest(INVENTORY)
print("Total corner×temp jobs:", manifest["count"])
print("Example job:", json.dumps(manifest["jobs"][0], indent=2))


## 10B.6 Synthesis — AnalogCoder loop as a pedagogical micro-MAS

**Takeaways:**

- **Stage A JSON** supplies *contracts* (PDK, targets, corners). **Stage B** is a **compiler + verifier** that *materializes* `.spice` programs.
- **Template-first** generation controls **topology safety**; **parameters** $\theta$ carry **continuous design freedom**.
- **Speclets** operationalize **domain prompts** per sub-block—they are the **human/LLM-readable** invariants your Python must respect.
- A **mock feedback loop** isolates **control-theoretic** questions (step size, weights, convergence) before **toolchain noise**.
- Replacing `mock_simulate_miller` with a **real Ngspice** wrapper yields the **same outer loop**, i.e. true **code-generating analog EDA**.

**Exercise extensions:**

1. Add **RZ** in series with `Cc` and extend $\theta$.
2. Replace finite-difference gradients with **automatic differentiation** on $\tilde{\mathcal{O}}$ if you make it fully smooth.
3. Instantiate a **second agent** that edits **SPECLETS** when violations persist (meta-level AnalogCoder).

---

*End of Chapter 10B notebook.*
